# Dates Generator — Four-Model Pipeline

Conditional date generation given (day-of-week, month, leap-year, decade) conditions.

**Models trained here (all from scratch):**
1. **CVAE** — Conditional VAE with a joint 310-way output head (in-course)
2. **CGAN** — Conditional GAN with Gumbel-Softmax + projection discriminator (in-course, required)
3. **AR Transformer** — tiny decoder-only LM, hardest-token-first ordering (out-of-course)
4. **Discrete Diffusion** — D3PM absorbing-state, T=10 (out-of-course)




In [3]:
# ============================================================================
# Setup
# ============================================================================
import os, sys, math, time, random, json, calendar, re, io, shutil, zipfile
from pathlib import Path
from collections import defaultdict, Counter
from dataclasses import dataclass
from typing import List, Tuple, Callable, Iterable

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt
%matplotlib inline

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__}")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


PyTorch: 2.10.0+cu128
Device: cuda
GPU: Tesla T4


## Configuration

Adjust the paths below for your Kaggle environment. If you uploaded `data.txt` as a Kaggle dataset named e.g. `dates-generator-data`, set `DATA_DIR = '/kaggle/input/dates-generator-data'`.


In [5]:
# ============================================================================
# Paths
# ============================================================================
# CHANGE THIS to wherever you uploaded data.txt + example_input.txt
DATA_DIR = "/kaggle/input/datasets/safeyelrahman/dates-generator-data"
# Where to write trained weights, predictions, plots, and the submission zip
OUT_DIR = Path("/kaggle/working")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Auto-detect if data is in /kaggle/working instead
for candidate in [DATA_DIR, "/kaggle/working", "/kaggle/input"]:
    if Path(candidate, "data.txt").exists():
        DATA_DIR = candidate
        break
    # Search one level deep under /kaggle/input
    for sub in Path("/kaggle/input").glob("*"):
        if (sub / "data.txt").exists():
            DATA_DIR = str(sub)
            break

DATA_PATH = Path(DATA_DIR) / "data.txt"
EXAMPLE_INPUT_PATH = Path(DATA_DIR) / "example_input.txt"

assert DATA_PATH.exists(), f"data.txt not found under {DATA_DIR}. Upload it as a Kaggle dataset."
assert EXAMPLE_INPUT_PATH.exists(), f"example_input.txt not found under {DATA_DIR}."
print(f"Using DATA_DIR = {DATA_DIR}")
print(f"data.txt: {DATA_PATH.stat().st_size / 1e6:.2f} MB")

WEIGHTS_DIR = OUT_DIR / "weights"
WEIGHTS_DIR.mkdir(exist_ok=True)
PLOTS_DIR = OUT_DIR / "plots"
PLOTS_DIR.mkdir(exist_ok=True)


Using DATA_DIR = /kaggle/input/datasets/safeyelrahman/dates-generator-data
data.txt: 5.23 MB


## 1. Tokenizer

Both encoding tracks supported:
- **Structured** (for CVAE, CGAN): condition → 4 indices, target → (day, year_units) joint index.
- **Token sequence** (for AR, Diffusion): full vocab of 41 tokens.

The output format on disk is `d-m-yyyy`. We predict only `year_units` and `day`; the month is fully determined by the input and is pasted through.


In [ ]:
# ============================================================================
# Tokenizer
# ============================================================================
DOW_TOKENS = ["[MON]", "[TUE]", "[WED]", "[THU]", "[FRI]", "[SAT]", "[SUN]"]
MON_TOKENS = ["[JAN]", "[FEB]", "[MAR]", "[APR]", "[MAY]", "[JUN]",
              "[JUL]", "[AUG]", "[SEP]", "[OCT]", "[NOV]", "[DEC]"]
LEAP_TOKENS = ["[False]", "[True]"]
DEC_TOKENS = [f"[{d}]" for d in range(180, 221)]

DOW_TO_IDX = {t: i for i, t in enumerate(DOW_TOKENS)}
MON_TO_IDX = {t: i for i, t in enumerate(MON_TOKENS)}
LEAP_TO_IDX = {t: i for i, t in enumerate(LEAP_TOKENS)}
DEC_TO_IDX = {t: i for i, t in enumerate(DEC_TOKENS)}
MON_NAME_TO_NUM = {t: i + 1 for i, t in enumerate(MON_TOKENS)}
NUM_TO_MON_NAME = {v: k for k, v in MON_NAME_TO_NUM.items()}

N_DOW, N_MON, N_LEAP, N_DEC = 7, 12, 2, 41
N_YEAR_UNITS = 10
N_DAY = 31

CONDITION_RE = re.compile(
    r"^\[(?P<dow>[A-Z]{3})\]\s+\[(?P<mon>[A-Z]{3})\]\s+"
    r"\[(?P<leap>True|False)\]\s+\[(?P<dec>\d{3})\]"
)


@dataclass(frozen=True)
class Condition:
    dow: str; mon: str; leap: str; dec: str

    def as_indices(self):
        return (DOW_TO_IDX[self.dow], MON_TO_IDX[self.mon],
                LEAP_TO_IDX[self.leap], DEC_TO_IDX[self.dec])

    @property
    def month_num(self): return MON_NAME_TO_NUM[self.mon]

    @property
    def decade_int(self): return int(self.dec.strip("[]"))

    def as_prefix(self): return f"{self.dow} {self.mon} {self.leap} {self.dec}"


def parse_condition_line(line: str) -> Condition:
    m = CONDITION_RE.match(line.strip())
    if not m:
        raise ValueError(f"Malformed: {line!r}")
    return Condition(f"[{m['dow']}]", f"[{m['mon']}]", f"[{m['leap']}]", f"[{m['dec']}]")


def parse_data_line(line: str):
    cond = parse_condition_line(line)
    d_s, m_s, y_s = line.strip().rsplit(maxsplit=1)[-1].split("-")
    return cond, int(d_s), int(m_s), int(y_s)


def format_output_line(cond: Condition, day: int, month: int, year: int) -> str:
    return f"{cond.as_prefix()} {day}-{month}-{year}"


# Token-sequence vocab for AR / diffusion
SPECIALS = ["<pad>", "<bos>", "<eos>", "<sep>", "<mask>"]
DIGIT_TOKENS = [str(d) for d in range(10)]
VOCAB = SPECIALS + DOW_TOKENS + MON_TOKENS + LEAP_TOKENS + DEC_TOKENS + DIGIT_TOKENS
TOKEN_TO_ID = {t: i for i, t in enumerate(VOCAB)}
ID_TO_TOKEN = {i: t for t, i in TOKEN_TO_ID.items()}
PAD_ID = TOKEN_TO_ID["<pad>"]
BOS_ID = TOKEN_TO_ID["<bos>"]
EOS_ID = TOKEN_TO_ID["<eos>"]
SEP_ID = TOKEN_TO_ID["<sep>"]
MASK_ID = TOKEN_TO_ID["<mask>"]
VOCAB_SIZE = len(VOCAB)
DIGIT_IDS = [TOKEN_TO_ID[str(d)] for d in range(10)]
SEQ_LEN = 10

print(f"Vocab size: {VOCAB_SIZE}")


## 2. Metrics — CSR (Condition Satisfaction Rate)

Pure functions, no model code. Every output we generate gets validated against these.


In [ ]:
# ============================================================================
# Metrics
# ============================================================================
def is_leap_year(year: int) -> bool:
    return (year % 4 == 0 and year % 100 != 0) or (year % 400 == 0)


def valid_date(day: int, month: int, year: int) -> bool:
    if not (1 <= month <= 12) or year < 1:
        return False
    return 1 <= day <= calendar.monthrange(year, month)[1]


def date_weekday_token(day: int, month: int, year: int) -> str:
    import datetime as _dt
    return DOW_TOKENS[_dt.date(year, month, day).weekday()]


def check_dow(d, m, y, c): return date_weekday_token(d, m, y) == c.dow
def check_mon(d, m, y, c): return NUM_TO_MON_NAME[m] == c.mon
def check_leap(d, m, y, c): return (c.leap == "[True]") == is_leap_year(y)
def check_dec(d, m, y, c):  return y // 10 == c.decade_int


@dataclass
class CSRReport:
    n: int; n_valid: int; valid_rate: float
    csr_all: float; csr_dow: float; csr_mon: float; csr_leap: float; csr_dec: float
    def __str__(self):
        return (f"n={self.n:<7d} valid={self.valid_rate:.3f}  "
                f"CSR_all={self.csr_all:.3f}  DOW={self.csr_dow:.3f}  "
                f"MON={self.csr_mon:.3f}  LEAP={self.csr_leap:.3f}  DEC={self.csr_dec:.3f}")


def csr_report(generations) -> CSRReport:
    n = len(generations)
    if n == 0:
        return CSRReport(0, 0, 0., 0., 0., 0., 0., 0.)
    n_valid = dow = mon = leap = dec = all_ok = 0
    for c, d, m, y in generations:
        if not valid_date(d, m, y):
            continue
        n_valid += 1
        a = check_dow(d, m, y, c); b = check_mon(d, m, y, c)
        e = check_leap(d, m, y, c); f = check_dec(d, m, y, c)
        dow += a; mon += b; leap += e; dec += f
        all_ok += a and b and e and f
    return CSRReport(n, n_valid, n_valid/n, all_ok/n, dow/n, mon/n, leap/n, dec/n)


def diversity_per_condition(generations):
    by_c = defaultdict(list)
    for c, d, m, y in generations:
        by_c[c].append((d, m, y))
    ratios = [len(set(s)) / len(s) for s in by_c.values() if len(s) >= 2]
    return (sum(ratios) / len(ratios), len(ratios)) if ratios else (0.0, 0)


## 3. Load data, sanity-check, build splits

If sanity check shows `CSR_all=1.000` over all rows, the parser and metrics agree with the data.


In [ ]:
# ============================================================================
# Load records
# ============================================================================
def load_records(path, frac=1.0, seed=0):
    records = []
    for line in Path(path).read_text().splitlines():
        if not line.strip():
            continue
        records.append(parse_data_line(line))
    if frac < 1.0:
        rng = random.Random(seed)
        k = int(len(records) * frac)
        records = rng.sample(records, k)
    return records


ALL_RECORDS = load_records(DATA_PATH)
print(f"Total rows: {len(ALL_RECORDS):,}")
print(f"Sanity check: {csr_report(ALL_RECORDS)}")


In [ ]:
# ============================================================================
# Build splits
# ============================================================================
def build_splits(records, seed=42, n_held_out_tuples=5, val_frac=0.05, test_frac=0.05):
    rng = random.Random(seed)
    by_tuple = defaultdict(list)
    for r in records:
        c = r[0]
        by_tuple[c.as_indices()].append(r)
    tuples = list(by_tuple.keys())
    rng.shuffle(tuples)
    held_keys = set(tuples[:n_held_out_tuples])

    held_out, remaining = [], []
    for r in records:
        (held_out if r[0].as_indices() in held_keys else remaining).append(r)
    rng.shuffle(remaining)
    n_val = int(len(remaining) * val_frac)
    n_test = int(len(remaining) * test_frac)
    val = remaining[:n_val]
    test_rand = remaining[n_val:n_val + n_test]
    train = remaining[n_val + n_test:]
    return train, val, test_rand, held_out


train_records, val_records, test_rand_records, test_held_records = build_splits(ALL_RECORDS)
print(f"Train: {len(train_records):,}  Val: {len(val_records):,}  "
      f"Test (random rows): {len(test_rand_records):,}  Test (held-out tuples): {len(test_held_records):,}")


## 4. Dataset classes


In [ ]:
# ============================================================================
# Dataset classes
# ============================================================================
def encode_struct_condition(c):
    return list(c.as_indices())


def encode_struct_target(d, m, y):
    return d - 1, m - 1, y % 10


def encode_token_sequence(c, day, year_units):
    d_t, d_u = divmod(day, 10)
    return [BOS_ID, TOKEN_TO_ID[c.dow], TOKEN_TO_ID[c.mon], TOKEN_TO_ID[c.leap],
            TOKEN_TO_ID[c.dec], SEP_ID,
            TOKEN_TO_ID[str(year_units)], TOKEN_TO_ID[str(d_t)],
            TOKEN_TO_ID[str(d_u)], EOS_ID]


class StructDataset(Dataset):
    def __init__(self, records):
        self.records = records
    def __len__(self): return len(self.records)
    def __getitem__(self, i):
        c, d, m, y = self.records[i]
        cond = torch.tensor(encode_struct_condition(c), dtype=torch.long)
        d_idx, _, yu_idx = encode_struct_target(d, m, y)
        return cond, torch.tensor(d_idx, dtype=torch.long), torch.tensor(yu_idx, dtype=torch.long)


class TokenSeqDataset(Dataset):
    def __init__(self, records):
        self.records = records
    def __len__(self): return len(self.records)
    def __getitem__(self, i):
        c, d, _, y = self.records[i]
        return torch.tensor(encode_token_sequence(c, d, y % 10), dtype=torch.long)


# Shared val eval helper -- works on a batched generator
@torch.no_grad()
def evaluate_batched(generate_batch_fn, records, max_n=None, chunk_size=1024):
    items = list(records) if max_n is None else list(records)[:max_n]
    conds = [r[0] for r in items]
    gens = []
    for s in range(0, len(conds), chunk_size):
        batch = conds[s:s + chunk_size]
        outs = generate_batch_fn(batch)
        for c, (d, m, y) in zip(batch, outs):
            gens.append((c, d, m, y))
    return csr_report(gens)


## 5. Baselines

`random` = uniform year_units + uniform valid day-in-month. The CSR floor.
`smart_random` = rejection sampling with calendar constraints. A brute-force ceiling that real models must beat through *learned* structure.


In [ ]:
# ============================================================================
# Baselines
# ============================================================================
def random_date(cond, rng):
    month = cond.month_num
    yu = rng.randint(0, 9)
    year = cond.decade_int * 10 + yu
    days = calendar.monthrange(year, month)[1]
    return rng.randint(1, days), month, year


def smart_random_date(cond, rng, max_tries=200):
    last = random_date(cond, rng)
    for _ in range(max_tries):
        d, m, y = random_date(cond, rng)
        if check_dow(d, m, y, cond) and check_leap(d, m, y, cond):
            return d, m, y
        last = (d, m, y)
    return last


def baseline_csr(gen_fn, records, label):
    rng = random.Random(0)
    gens = [(c, *gen_fn(c, rng)) for c, _, _, _ in records[:2000]]
    rep = csr_report(gens)
    print(f"  {label:<14s} {rep}")
    return rep


print("Baselines on val set (first 2000 records):")
random_rep = baseline_csr(random_date, val_records, "random")
smart_rep = baseline_csr(smart_random_date, val_records, "smart_random")


## 6. Shared training helpers


In [ ]:
# ============================================================================
# Shared training helpers
# ============================================================================
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


class ConditionEmbedder(nn.Module):
    def __init__(self, embed_dim=32):
        super().__init__()
        self.dow = nn.Embedding(N_DOW, embed_dim)
        self.mon = nn.Embedding(N_MON, embed_dim)
        self.leap = nn.Embedding(N_LEAP, embed_dim)
        self.dec = nn.Embedding(N_DEC, embed_dim)
        self.out_dim = 4 * embed_dim

    def forward(self, cond_idx):
        return torch.cat([self.dow(cond_idx[:, 0]),
                          self.mon(cond_idx[:, 1]),
                          self.leap(cond_idx[:, 2]),
                          self.dec(cond_idx[:, 3])], dim=-1)


HISTORY = {}  # model_name -> list of {"epoch", "loss", "val_csr_all", ...}


## 7. Model 1 — Conditional VAE

Encoder takes (condition + true day/year_units) and produces a latent z.
Decoder takes z + condition and produces a **joint** logit over (day × year_units) = 310 classes.
The joint head captures the dependency that determines DOW.

Training: β-warmup + free-bits to prevent posterior collapse.


In [ ]:
# ============================================================================
# Model 1: CVAE
# ============================================================================
class CVAE(nn.Module):
    def __init__(self, latent_dim=32, hidden=512, embed_dim=32):
        super().__init__()
        self.latent_dim = latent_dim
        self.cond_embed = ConditionEmbedder(embed_dim)
        cond_dim = self.cond_embed.out_dim

        enc_in = cond_dim + N_DAY + N_YEAR_UNITS
        self.encoder = nn.Sequential(
            nn.Linear(enc_in, hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.GELU(),
        )
        self.fc_mu = nn.Linear(hidden, latent_dim)
        self.fc_logvar = nn.Linear(hidden, latent_dim)

        self.decoder_trunk = nn.Sequential(
            nn.Linear(latent_dim + cond_dim, hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.GELU(),
        )
        self.joint_head = nn.Linear(hidden, N_DAY * N_YEAR_UNITS)

    def encode(self, cond_idx, d_idx, yu_idx):
        c = self.cond_embed(cond_idx)
        d_oh = F.one_hot(d_idx, N_DAY).float()
        y_oh = F.one_hot(yu_idx, N_YEAR_UNITS).float()
        h = self.encoder(torch.cat([c, d_oh, y_oh], dim=-1))
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + std * torch.randn_like(std)

    def decode(self, z, cond_idx):
        c = self.cond_embed(cond_idx)
        h = self.decoder_trunk(torch.cat([z, c], dim=-1))
        return self.joint_head(h)

    def forward(self, cond_idx, d_idx, yu_idx):
        mu, logvar = self.encode(cond_idx, d_idx, yu_idx)
        z = self.reparameterize(mu, logvar)
        return self.decode(z, cond_idx), mu, logvar

    @torch.no_grad()
    def sample(self, cond_idx):
        z = torch.randn(cond_idx.size(0), self.latent_dim, device=cond_idx.device)
        joint_logits = self.decode(z, cond_idx)
        joint_idx = torch.distributions.Categorical(logits=joint_logits).sample()
        return joint_idx // N_YEAR_UNITS, joint_idx % N_YEAR_UNITS


def cvae_batch_generate(model, conds):
    idx = torch.tensor([encode_struct_condition(c) for c in conds], dtype=torch.long, device=DEVICE)
    d_idx, yu_idx = model.sample(idx)
    outs = []
    for i, c in enumerate(conds):
        outs.append((d_idx[i].item() + 1, c.month_num, c.decade_int * 10 + yu_idx[i].item()))
    return outs


def train_cvae(epochs=12, batch_size=1024, lr=1e-3, latent_dim=32, hidden=512,
               beta_warmup_epochs=3, free_bits=0.02, val_eval_n=2000):
    ds = StructDataset(train_records)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=2, drop_last=True)
    model = CVAE(latent_dim=latent_dim, hidden=hidden).to(DEVICE)
    print(f"[cvae] params: {count_params(model):,}")
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    history = []
    best_csr = -1.0

    for epoch in range(epochs):
        model.train()
        beta = min(1.0, (epoch + 1) / max(1, beta_warmup_epochs))
        t0 = time.time()
        rl = rr = rk = 0.; nb = 0
        for cond, d_idx, yu_idx in loader:
            cond = cond.to(DEVICE); d_idx = d_idx.to(DEVICE); yu_idx = yu_idx.to(DEVICE)
            joint_logits, mu, logvar = model(cond, d_idx, yu_idx)
            joint_idx = d_idx * N_YEAR_UNITS + yu_idx
            recon = F.cross_entropy(joint_logits, joint_idx)
            kl_per_dim = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp())
            kl_per_dim = torch.clamp(kl_per_dim.mean(dim=0), min=free_bits)
            kl = kl_per_dim.sum()
            loss = recon + beta * kl
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            rl += loss.item(); rr += recon.item(); rk += kl.item(); nb += 1

        model.eval()
        rep = evaluate_batched(lambda cs: cvae_batch_generate(model, cs),
                                val_records, max_n=val_eval_n)
        history.append({"epoch": epoch, "loss": rl/nb, "recon": rr/nb, "kl": rk/nb,
                        "beta": beta, "val_csr_all": rep.csr_all,
                        "val_csr_dow": rep.csr_dow, "val_csr_leap": rep.csr_leap})
        print(f"[cvae] ep{epoch:>2d} loss={rl/nb:.4f} recon={rr/nb:.4f} kl={rk/nb:.4f} "
              f"beta={beta:.2f} val_CSR_all={rep.csr_all:.3f} "
              f"(DOW={rep.csr_dow:.3f} LEAP={rep.csr_leap:.3f}) [{time.time()-t0:.1f}s]")
        if rep.csr_all > best_csr:
            best_csr = rep.csr_all
            torch.save({"state_dict": model.state_dict(),
                        "cfg": {"latent_dim": latent_dim, "hidden": hidden},
                        "history": history, "best_csr": best_csr},
                       WEIGHTS_DIR / "cvae.pt")
    return model, history


print("\n===== Training CVAE =====")
cvae_model, cvae_history = train_cvae()
HISTORY["cvae"] = cvae_history


## 8. Model 2 — Autoregressive Transformer

Tiny decoder-only model over the token sequence. Predict order: `year_units → day_tens → day_units` (month is fixed by the input and pasted through). Hardest token first.


In [ ]:
# ============================================================================
# Model 2: AR Transformer
# ============================================================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=SEQ_LEN):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x): return x + self.pe[:, :x.size(1)]


class ARTransformer(nn.Module):
    def __init__(self, d_model=128, n_layers=4, n_heads=4, ff_dim=256, dropout=0.1):
        super().__init__()
        self.embed = nn.Embedding(VOCAB_SIZE, d_model)
        self.pos = PositionalEncoding(d_model)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=ff_dim,
            dropout=dropout, batch_first=True, activation="gelu", norm_first=True)
        self.backbone = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.head = nn.Linear(d_model, VOCAB_SIZE)
        mask = torch.triu(torch.ones(SEQ_LEN - 1, SEQ_LEN - 1), diagonal=1).bool()
        self.register_buffer("causal_mask", mask)
    def forward(self, ids):
        h = self.pos(self.embed(ids))
        h = self.backbone(h, mask=self.causal_mask, is_causal=True)
        return self.head(h)


def sample_top_p(logits, temperature=1.0, top_p=0.9):
    if temperature != 1.0:
        logits = logits / temperature
    probs = F.softmax(logits, dim=-1)
    if top_p < 1.0:
        sp, si = torch.sort(probs, descending=True, dim=-1)
        cu = torch.cumsum(sp, dim=-1)
        cut = cu > top_p
        cut[..., 1:] = cut[..., :-1].clone()
        cut[..., 0] = False
        sp[cut] = 0.0
        sp = sp / sp.sum(dim=-1, keepdim=True)
        s = torch.multinomial(sp, num_samples=1).squeeze(-1)
        return si.gather(-1, s.unsqueeze(-1)).squeeze(-1)
    return torch.multinomial(probs, num_samples=1).squeeze(-1)


@torch.no_grad()
def ar_generate_batch(model, conds, temperature=1.0, top_p=0.9):
    B = len(conds)
    if B == 0:
        return []
    ids = torch.full((B, SEQ_LEN - 1), PAD_ID, dtype=torch.long, device=DEVICE)
    for i, c in enumerate(conds):
        ids[i, 0] = BOS_ID
        ids[i, 1] = TOKEN_TO_ID[c.dow]
        ids[i, 2] = TOKEN_TO_ID[c.mon]
        ids[i, 3] = TOKEN_TO_ID[c.leap]
        ids[i, 4] = TOKEN_TO_ID[c.dec]
        ids[i, 5] = SEP_ID
    digit_ids_t = torch.tensor(DIGIT_IDS, dtype=torch.long, device=DEVICE)
    for write_pos in (6, 7, 8):
        logits = model(ids)
        row_logits = logits[:, write_pos - 1, :]
        digit_logits = row_logits[:, DIGIT_IDS]
        sampled = sample_top_p(digit_logits, temperature, top_p)
        ids[:, write_pos] = digit_ids_t[sampled]
    outs = []
    for i, c in enumerate(conds):
        y_u = int(ID_TO_TOKEN[ids[i, 6].item()])
        d_t = int(ID_TO_TOKEN[ids[i, 7].item()])
        d_u = int(ID_TO_TOKEN[ids[i, 8].item()])
        outs.append((d_t * 10 + d_u, c.month_num, c.decade_int * 10 + y_u))
    return outs


def train_ar(epochs=12, batch_size=1024, lr=3e-3, d_model=128, n_layers=4,
             n_heads=4, ff_dim=256, val_eval_n=2000):
    ds = TokenSeqDataset(train_records)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=2, drop_last=True)
    model = ARTransformer(d_model=d_model, n_layers=n_layers, n_heads=n_heads, ff_dim=ff_dim).to(DEVICE)
    print(f"[ar] params: {count_params(model):,}")
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    pos_t = torch.tensor([5, 6, 7], dtype=torch.long, device=DEVICE)
    history = []; best_csr = -1.0

    for epoch in range(epochs):
        model.train()
        t0 = time.time(); rl = 0.; nb = 0
        for seq in loader:
            seq = seq.to(DEVICE)
            inp = seq[:, :-1]; tgt = seq[:, 1:]
            logits = model(inp)
            sliced_logits = logits[:, pos_t, :]
            sliced_tgt = tgt[:, pos_t]
            loss = F.cross_entropy(sliced_logits.reshape(-1, VOCAB_SIZE), sliced_tgt.reshape(-1))
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            rl += loss.item(); nb += 1
        model.eval()
        rep = evaluate_batched(lambda cs: ar_generate_batch(model, cs), val_records, max_n=val_eval_n)
        history.append({"epoch": epoch, "loss": rl/nb, "val_csr_all": rep.csr_all,
                        "val_csr_dow": rep.csr_dow, "val_csr_leap": rep.csr_leap})
        print(f"[ar] ep{epoch:>2d} loss={rl/nb:.4f} val_CSR_all={rep.csr_all:.3f} "
              f"(DOW={rep.csr_dow:.3f} LEAP={rep.csr_leap:.3f}) [{time.time()-t0:.1f}s]")
        if rep.csr_all > best_csr:
            best_csr = rep.csr_all
            torch.save({"state_dict": model.state_dict(),
                        "cfg": {"d_model": d_model, "n_layers": n_layers,
                                "n_heads": n_heads, "ff_dim": ff_dim},
                        "history": history, "best_csr": best_csr},
                       WEIGHTS_DIR / "ar.pt")
    return model, history


print("\n===== Training AR Transformer =====")
ar_model, ar_history = train_ar()
HISTORY["ar"] = ar_history


## 9. Model 3 — Conditional GAN

Required by the spec (must be one of the two course models).

Discrete outputs + adversarial training is notoriously brittle. We use:
- **Gumbel-Softmax** with temperature annealing (1.0 → 0.1) for differentiable discrete G outputs
- **Joint 310-way head** on G (same shape as CVAE)
- **Projection discriminator** for stable conditioning (`w·φ(x) + e(cond)·φ(x)`)
- **Spectral normalization** on D's linear layers
- **Hinge loss**

Don't expect this to match the AR Transformer; discrete GANs are inherently harder than likelihood-based models on this task. A CSR around 0.5–0.8 is a respectable result.


In [ ]:
# ============================================================================
# Model 3: CGAN
# ============================================================================
N_JOINT = N_DAY * N_YEAR_UNITS  # 310


class GeneratorNet(nn.Module):
    def __init__(self, noise_dim=32, hidden=256, embed_dim=32):
        super().__init__()
        self.noise_dim = noise_dim
        self.cond_embed = ConditionEmbedder(embed_dim)
        self.trunk = nn.Sequential(
            nn.Linear(noise_dim + self.cond_embed.out_dim, hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.GELU(),
        )
        self.head = nn.Linear(hidden, N_JOINT)

    def forward(self, z, cond_idx, tau=1.0, hard=False):
        c = self.cond_embed(cond_idx)
        h = self.trunk(torch.cat([z, c], dim=-1))
        logits = self.head(h)
        return F.gumbel_softmax(logits, tau=tau, hard=hard, dim=-1)


def sn(m):
    return nn.utils.parametrizations.spectral_norm(m)


class DiscriminatorNet(nn.Module):
    # Projection discriminator: real/fake = w.phi(x) + e(c).phi(x).

    def __init__(self, hidden=256, embed_dim=32):
        super().__init__()
        self.cond_embed = ConditionEmbedder(embed_dim)
        self.trunk = nn.Sequential(
            sn(nn.Linear(N_JOINT, hidden)), nn.LeakyReLU(0.2),
            sn(nn.Linear(hidden, hidden)), nn.LeakyReLU(0.2),
            sn(nn.Linear(hidden, hidden)), nn.LeakyReLU(0.2),
        )
        self.real_fake = sn(nn.Linear(hidden, 1))
        self.proj = sn(nn.Linear(self.cond_embed.out_dim, hidden))

    def forward(self, x_soft, cond_idx):
        phi = self.trunk(x_soft)
        rf = self.real_fake(phi).squeeze(-1)
        cond_emb = self.cond_embed(cond_idx)
        proj = (self.proj(cond_emb) * phi).sum(-1)
        return rf + proj


def cgan_batch_generate(model_G, conds, tau=0.1):
    idx = torch.tensor([encode_struct_condition(c) for c in conds], dtype=torch.long, device=DEVICE)
    z = torch.randn(idx.size(0), model_G.noise_dim, device=DEVICE)
    with torch.no_grad():
        soft = model_G(z, idx, tau=tau, hard=True)
    sampled = soft.argmax(dim=-1)
    d_idx = sampled // N_YEAR_UNITS
    yu_idx = sampled % N_YEAR_UNITS
    return [(d_idx[i].item() + 1, c.month_num, c.decade_int * 10 + yu_idx[i].item())
            for i, c in enumerate(conds)]


def train_cgan(epochs=20, batch_size=1024, lr=2e-4, noise_dim=32, hidden=256,
               tau_start=1.0, tau_end=0.3, val_eval_n=1500):
    ds = StructDataset(train_records)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=2, drop_last=True)
    G = GeneratorNet(noise_dim=noise_dim, hidden=hidden).to(DEVICE)
    D = DiscriminatorNet(hidden=hidden).to(DEVICE)
    print(f"[cgan] G params: {count_params(G):,}  D params: {count_params(D):,}")
    optG = torch.optim.AdamW(G.parameters(), lr=lr, betas=(0.5, 0.9))
    optD = torch.optim.AdamW(D.parameters(), lr=lr, betas=(0.5, 0.9))
    history = []; best_csr = -1.0

    for epoch in range(epochs):
        G.train(); D.train()
        # Anneal tau linearly
        tau = tau_start + (tau_end - tau_start) * (epoch / max(1, epochs - 1))
        t0 = time.time()
        rl_d = rl_g = 0.; nb = 0
        for cond, d_idx, yu_idx in loader:
            cond = cond.to(DEVICE); d_idx = d_idx.to(DEVICE); yu_idx = yu_idx.to(DEVICE)
            B = cond.size(0)
            joint_real = d_idx * N_YEAR_UNITS + yu_idx
            x_real = F.one_hot(joint_real, N_JOINT).float()

            # ---- D step (hinge) ----
            z = torch.randn(B, noise_dim, device=DEVICE)
            x_fake = G(z, cond, tau=tau, hard=False).detach()
            d_real = D(x_real, cond)
            d_fake = D(x_fake, cond)
            d_loss = F.relu(1.0 - d_real).mean() + F.relu(1.0 + d_fake).mean()
            optD.zero_grad(); d_loss.backward()
            torch.nn.utils.clip_grad_norm_(D.parameters(), 5.0); optD.step()

            # ---- G step (hinge) ----
            z = torch.randn(B, noise_dim, device=DEVICE)
            x_fake = G(z, cond, tau=tau, hard=False)
            g_loss = -D(x_fake, cond).mean()
            optG.zero_grad(); g_loss.backward()
            torch.nn.utils.clip_grad_norm_(G.parameters(), 5.0); optG.step()

            rl_d += d_loss.item(); rl_g += g_loss.item(); nb += 1

        G.eval()
        rep = evaluate_batched(lambda cs: cgan_batch_generate(G, cs, tau=tau_end),
                                val_records, max_n=val_eval_n)
        history.append({"epoch": epoch, "d_loss": rl_d/nb, "g_loss": rl_g/nb, "tau": tau,
                        "val_csr_all": rep.csr_all, "val_csr_dow": rep.csr_dow,
                        "val_csr_leap": rep.csr_leap})
        print(f"[cgan] ep{epoch:>2d} d_loss={rl_d/nb:.4f} g_loss={rl_g/nb:.4f} "
              f"tau={tau:.2f} val_CSR_all={rep.csr_all:.3f} "
              f"(DOW={rep.csr_dow:.3f} LEAP={rep.csr_leap:.3f}) [{time.time()-t0:.1f}s]")
        if rep.csr_all > best_csr:
            best_csr = rep.csr_all
            torch.save({"state_dict_G": G.state_dict(), "state_dict_D": D.state_dict(),
                        "cfg": {"noise_dim": noise_dim, "hidden": hidden, "tau_end": tau_end},
                        "history": history, "best_csr": best_csr},
                       WEIGHTS_DIR / "cgan.pt")
    return G, D, history


print("\n===== Training CGAN =====")
cgan_G, cgan_D, cgan_history = train_cgan()
HISTORY["cgan"] = cgan_history


## 10. Model 4 — Discrete Diffusion (D3PM, absorbing-state)

Two discrete tokens: `year_units ∈ {0..9, mask}` and `day ∈ {1..31, mask}`.

**Forward**: at step *t* ∈ [1, T], each token is independently masked with prob *t/T*.
**Reverse**: a denoising MLP predicts `p(x₀ | x_t, t, cond)` and we iteratively unmask.

Sampling: start from (mask, mask); at each step from t=T down to 1, for any still-masked position, with probability 1/t we sample x₀ and unmask it.


In [ ]:
# ============================================================================
# Model 4: Discrete Diffusion (D3PM absorbing-state)
# ============================================================================
T_STEPS = 10
MASK_DAY_IDX = N_DAY        # extra "mask" index for day
MASK_YU_IDX = N_YEAR_UNITS  # extra "mask" index for year_units


class DiffusionNet(nn.Module):
    def __init__(self, hidden=512, embed_dim=32, T=T_STEPS):
        super().__init__()
        self.cond_embed = ConditionEmbedder(embed_dim)
        self.day_embed = nn.Embedding(N_DAY + 1, embed_dim)
        self.yu_embed = nn.Embedding(N_YEAR_UNITS + 1, embed_dim)
        self.t_embed = nn.Embedding(T + 1, embed_dim)
        in_dim = self.cond_embed.out_dim + 2 * embed_dim + embed_dim
        self.trunk = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.GELU(),
        )
        self.day_head = nn.Linear(hidden, N_DAY)        # predict 1..31 (no mask in output)
        self.yu_head = nn.Linear(hidden, N_YEAR_UNITS)  # predict 0..9

    def forward(self, d_idx, yu_idx, cond_idx, t):
        c = self.cond_embed(cond_idx)
        d = self.day_embed(d_idx)
        y = self.yu_embed(yu_idx)
        te = self.t_embed(t)
        h = self.trunk(torch.cat([c, d, y, te], dim=-1))
        return self.day_head(h), self.yu_head(h)


@torch.no_grad()
def diffusion_batch_generate(model, conds, T=T_STEPS):
    B = len(conds)
    if B == 0:
        return []
    cond_idx = torch.tensor([encode_struct_condition(c) for c in conds], dtype=torch.long, device=DEVICE)
    d_idx = torch.full((B,), MASK_DAY_IDX, dtype=torch.long, device=DEVICE)
    yu_idx = torch.full((B,), MASK_YU_IDX, dtype=torch.long, device=DEVICE)
    for t in range(T, 0, -1):
        tt = torch.full((B,), t, dtype=torch.long, device=DEVICE)
        d_logits, yu_logits = model(d_idx, yu_idx, cond_idx, tt)
        d_sample = torch.distributions.Categorical(logits=d_logits).sample()
        yu_sample = torch.distributions.Categorical(logits=yu_logits).sample()
        d_masked = d_idx == MASK_DAY_IDX
        yu_masked = yu_idx == MASK_YU_IDX
        unmask_prob = 1.0 / t
        d_unmask = (torch.rand(B, device=DEVICE) < unmask_prob) & d_masked
        yu_unmask = (torch.rand(B, device=DEVICE) < unmask_prob) & yu_masked
        d_idx = torch.where(d_unmask, d_sample, d_idx)
        yu_idx = torch.where(yu_unmask, yu_sample, yu_idx)
    # Snap any still-masked positions (rare) by sampling from t=1 logits
    if (d_idx == MASK_DAY_IDX).any() or (yu_idx == MASK_YU_IDX).any():
        tt = torch.ones(B, dtype=torch.long, device=DEVICE)
        d_logits, yu_logits = model(d_idx, yu_idx, cond_idx, tt)
        d_sample = torch.distributions.Categorical(logits=d_logits).sample()
        yu_sample = torch.distributions.Categorical(logits=yu_logits).sample()
        d_idx = torch.where(d_idx == MASK_DAY_IDX, d_sample, d_idx)
        yu_idx = torch.where(yu_idx == MASK_YU_IDX, yu_sample, yu_idx)
    return [(d_idx[i].item() + 1, c.month_num, c.decade_int * 10 + yu_idx[i].item())
            for i, c in enumerate(conds)]


def train_diffusion(epochs=12, batch_size=1024, lr=1e-3, hidden=512,
                     T=T_STEPS, val_eval_n=2000):
    ds = StructDataset(train_records)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=2, drop_last=True)
    model = DiffusionNet(hidden=hidden, T=T).to(DEVICE)
    print(f"[diffusion] params: {count_params(model):,}")
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    history = []; best_csr = -1.0

    for epoch in range(epochs):
        model.train(); t0 = time.time(); rl = 0.; nb = 0
        for cond, d_idx, yu_idx in loader:
            cond = cond.to(DEVICE); d_idx = d_idx.to(DEVICE); yu_idx = yu_idx.to(DEVICE)
            B = cond.size(0)
            t = torch.randint(1, T + 1, (B,), device=DEVICE)
            p_mask = t.float() / T
            mask_d = torch.rand(B, device=DEVICE) < p_mask
            mask_y = torch.rand(B, device=DEVICE) < p_mask
            d_in = torch.where(mask_d, torch.full_like(d_idx, MASK_DAY_IDX), d_idx)
            y_in = torch.where(mask_y, torch.full_like(yu_idx, MASK_YU_IDX), yu_idx)

            d_logits, yu_logits = model(d_in, y_in, cond, t)
            # CE only on masked positions
            loss_d = (F.cross_entropy(d_logits, d_idx, reduction="none") * mask_d.float()).sum() / mask_d.float().sum().clamp(min=1.0)
            loss_y = (F.cross_entropy(yu_logits, yu_idx, reduction="none") * mask_y.float()).sum() / mask_y.float().sum().clamp(min=1.0)
            loss = loss_d + loss_y
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            rl += loss.item(); nb += 1

        model.eval()
        rep = evaluate_batched(lambda cs: diffusion_batch_generate(model, cs, T=T),
                                val_records, max_n=val_eval_n)
        history.append({"epoch": epoch, "loss": rl/nb, "val_csr_all": rep.csr_all,
                        "val_csr_dow": rep.csr_dow, "val_csr_leap": rep.csr_leap})
        print(f"[diffusion] ep{epoch:>2d} loss={rl/nb:.4f} val_CSR_all={rep.csr_all:.3f} "
              f"(DOW={rep.csr_dow:.3f} LEAP={rep.csr_leap:.3f}) [{time.time()-t0:.1f}s]")
        if rep.csr_all > best_csr:
            best_csr = rep.csr_all
            torch.save({"state_dict": model.state_dict(),
                        "cfg": {"hidden": hidden, "T": T},
                        "history": history, "best_csr": best_csr},
                       WEIGHTS_DIR / "diffusion.pt")
    return model, history


print("\n===== Training Discrete Diffusion =====")
diff_model, diff_history = train_diffusion()
HISTORY["diffusion"] = diff_history


## 11. Final evaluation across all models

CSR breakdown on:
- `val` (in-distribution)
- `test_random` (held-out rows, in-distribution conditions)
- `test_held_out_tuples` (OOD condition tuples — model has never seen these inputs)

Plus per-model diversity (`unique_rate` over k=32 samples per condition).


In [ ]:
# ============================================================================
# Final evaluation
# ============================================================================
def reload_best_models():
    # Reload best checkpoints (we saved on best val CSR throughout training)
    ck = torch.load(WEIGHTS_DIR / "cvae.pt", map_location=DEVICE, weights_only=False)
    m_cvae = CVAE(**ck["cfg"]).to(DEVICE); m_cvae.load_state_dict(ck["state_dict"]); m_cvae.eval()

    ck = torch.load(WEIGHTS_DIR / "ar.pt", map_location=DEVICE, weights_only=False)
    m_ar = ARTransformer(**ck["cfg"]).to(DEVICE); m_ar.load_state_dict(ck["state_dict"]); m_ar.eval()

    ck = torch.load(WEIGHTS_DIR / "cgan.pt", map_location=DEVICE, weights_only=False)
    cgan_cfg = ck["cfg"]
    m_cgan_G = GeneratorNet(noise_dim=cgan_cfg["noise_dim"], hidden=cgan_cfg["hidden"]).to(DEVICE)
    m_cgan_G.load_state_dict(ck["state_dict_G"]); m_cgan_G.eval()

    ck = torch.load(WEIGHTS_DIR / "diffusion.pt", map_location=DEVICE, weights_only=False)
    m_diff = DiffusionNet(hidden=ck["cfg"]["hidden"], T=ck["cfg"]["T"]).to(DEVICE)
    m_diff.load_state_dict(ck["state_dict"]); m_diff.eval()
    return m_cvae, m_ar, m_cgan_G, m_diff


cvae_b, ar_b, cgan_G_b, diff_b = reload_best_models()

GENERATORS = {
    "random":       lambda cs: [random_date(c, random.Random()) for c in cs],
    "smart_random": lambda cs: [smart_random_date(c, random.Random()) for c in cs],
    "cvae":         lambda cs: cvae_batch_generate(cvae_b, cs),
    "cgan":         lambda cs: cgan_batch_generate(cgan_G_b, cs),
    "ar":           lambda cs: ar_generate_batch(ar_b, cs),
    "diffusion":    lambda cs: diffusion_batch_generate(diff_b, cs),
}


print("\n========= FINAL CSR =========\n")
final_results = {}
for split_name, split_records in [("val", val_records),
                                   ("test_random", test_rand_records),
                                   ("test_held_out_tuples", test_held_records)]:
    print(f"--- {split_name} (n={len(split_records)}) ---")
    final_results[split_name] = {}
    for model_name, gen_fn in GENERATORS.items():
        rep = evaluate_batched(gen_fn, split_records)
        final_results[split_name][model_name] = rep
        print(f"  {model_name:<14s} {rep}")
    print()


In [ ]:
# ============================================================================
# Diversity check (k=32 samples per condition)
# ============================================================================
def diversity_check(gen_fn, conds_unique, k=32):
    # Replicate each condition k times
    replicated = []
    for c in conds_unique:
        replicated.extend([c] * k)
    outs = gen_fn(replicated)
    pairs = list(zip(replicated, outs))
    by_c = defaultdict(list)
    for c, (d, m, y) in pairs:
        by_c[c].append((d, m, y))
    ratios = [len(set(s)) / len(s) for s in by_c.values()]
    return sum(ratios) / len(ratios)


# Sample 200 unique conditions from val
unique_val_conds = list({r[0] for r in val_records})[:200]
print("\n========= DIVERSITY (mean unique-rate over k=32 samples per condition) =========\n")
for model_name, gen_fn in [("cvae", GENERATORS["cvae"]),
                            ("cgan", GENERATORS["cgan"]),
                            ("ar", GENERATORS["ar"]),
                            ("diffusion", GENERATORS["diffusion"])]:
    div = diversity_check(gen_fn, unique_val_conds, k=32)
    print(f"  {model_name:<14s} unique_rate={div:.3f}  (1.0 = all unique, 0.03 = always the same)")


## 12. Plots — training curves & per-model CSR comparison


In [ ]:
# ============================================================================
# Plot training curves
# ============================================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for name in ["cvae", "cgan", "ar", "diffusion"]:
    h = HISTORY[name]
    epochs = [e["epoch"] for e in h]
    csr = [e["val_csr_all"] for e in h]
    axes[0].plot(epochs, csr, marker="o", label=name)
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("val CSR_all"); axes[0].set_title("Validation CSR over training")
axes[0].legend(); axes[0].grid(alpha=0.3); axes[0].set_ylim(0, 1.05)

for name in ["cvae", "ar", "diffusion"]:
    h = HISTORY[name]
    epochs = [e["epoch"] for e in h]
    losses = [e["loss"] for e in h]
    axes[1].plot(epochs, losses, marker="o", label=name)
# CGAN has two losses
h = HISTORY["cgan"]
epochs = [e["epoch"] for e in h]
axes[1].plot(epochs, [e["g_loss"] for e in h], marker="x", linestyle="--", label="cgan-G")
axes[1].plot(epochs, [e["d_loss"] for e in h], marker="x", linestyle=":", label="cgan-D")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("loss"); axes[1].set_title("Training loss")
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "training_curves.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# ============================================================================
# Plot final CSR comparison
# ============================================================================
model_names = ["random", "smart_random", "cvae", "cgan", "ar", "diffusion"]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (split_name, results) in zip(axes, final_results.items()):
    xs = np.arange(len(model_names))
    csr_all = [results[m].csr_all for m in model_names]
    bars = ax.bar(xs, csr_all)
    for x, v in zip(xs, csr_all):
        ax.text(x, v + 0.01, f"{v:.2f}", ha="center", fontsize=9)
    ax.set_xticks(xs); ax.set_xticklabels(model_names, rotation=30, ha="right")
    ax.set_ylim(0, 1.1); ax.set_ylabel("CSR_all")
    ax.set_title(f"{split_name}  (n={results['cvae'].n})")
    ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "csr_comparison.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# ============================================================================
# Per-condition CSR breakdown for the best learned model
# ============================================================================
# Find the model with the best val CSR_all
best_model_name = max(["cvae", "cgan", "ar", "diffusion"],
                       key=lambda m: final_results["val"][m].csr_all)
print(f"Best learned model on val: {best_model_name}")

fig, ax = plt.subplots(figsize=(9, 4))
conditions = ["CSR_all", "DOW", "MON", "LEAP", "DEC"]
xs = np.arange(len(model_names))
width = 0.16
for i, cond_name in enumerate(conditions):
    vals = [getattr(final_results["val"][m], "csr_" + cond_name.lower().replace("csr_", "")) for m in model_names]
    ax.bar(xs + (i - 2) * width, vals, width, label=cond_name)
ax.set_xticks(xs); ax.set_xticklabels(model_names, rotation=30, ha="right")
ax.set_ylim(0, 1.1); ax.set_ylabel("rate")
ax.set_title("Per-condition breakdown on val set")
ax.legend(loc="lower right", fontsize=9); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "per_condition_breakdown.png", dpi=120, bbox_inches="tight")
plt.show()


## 13. Generate predictions on `example_input.txt`

Writes one predictions file per model, in the exact on-disk format the spec demands.


In [ ]:
# ============================================================================
# Generate predictions
# ============================================================================
input_lines = [ln for ln in EXAMPLE_INPUT_PATH.read_text().splitlines() if ln.strip()]
input_conds = [parse_condition_line(ln) for ln in input_lines]
print(f"Input lines: {len(input_conds)}")

PREDICTIONS_DIR = OUT_DIR / "predictions"
PREDICTIONS_DIR.mkdir(exist_ok=True)
for model_name, gen_fn in GENERATORS.items():
    outs = gen_fn(input_conds)
    lines = [format_output_line(c, d, m, y) for c, (d, m, y) in zip(input_conds, outs)]
    (PREDICTIONS_DIR / f"predictions_{model_name}.txt").write_text("\n".join(lines) + "\n")
    rep = csr_report([(c, d, m, y) for c, (d, m, y) in zip(input_conds, outs)])
    print(f"  {model_name:<14s} {rep}")
